In [23]:
# ## install finrl library
!pip install wrds
!pip install swig
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install finrl



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to c:\users\email\appdata\local\temp\pip-install-vllwjs6i\elegantrl_fb6b188736fb4510a07c482f6db75e14
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 62bf617fa87cf82b9fbf88307715d93b8ab8ee80
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git 'C:\Users\email\AppData\Local\Temp\pip-install-vllwjs6i\elegantrl_fb6b188736fb4510a07c482f6db75e14'

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import warnings
warnings.filterwarnings("ignore")

In [25]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

import gym
from finrl import config
from finrl.agents.stablebaselines3.models import DRLEnsembleAgent
from finrl.meta.preprocessor.preprocessors import FeatureEngineer

from pprint import pprint

import sys
sys.path.append("../FinRL-Library")

import itertools

In [26]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

In [27]:
TRAIN_START_DATE = '2017-01-01'
TRAIN_END_DATE = '2021-10-01'
TEST_START_DATE = '2021-10-01'
TEST_END_DATE = '2023-03-01'

In [28]:
df = pd.read_csv("C:\\Users\\email\\Downloads\\historical_data.csv")

In [29]:
#Data Exploration
print("First 5 rows of the dataset:")
print(df.head())

First 5 rows of the dataset:
         date      close       high        low       open  volume     tic  day
0  2004-01-01   0.105000   0.105000   0.105000   0.105000       0  DYA.TO    3
1  2004-01-01   1.378000   1.378000   1.378000   1.378000       0  EDT.TO    3
2  2004-01-01   0.300000   0.300000   0.300000   0.300000       0  TSK.TO    3
3  2004-01-02   0.489625   0.489625   0.489625   0.489625       0  AAB.TO    4
4  2004-01-02  22.070263  22.211644  21.817266  21.899118  413000  ABX.TO    4


In [30]:
print(df.isnull().sum())  # Shows the count of missing values per column

date      0
close     0
high      0
low       0
open      0
volume    0
tic       0
day       0
dtype: int64


In [31]:
# Convert 'date' column to datetime if it is not already
df['date'] = pd.to_datetime(df['date']) 

# *Step 1: Data Cleaning*
df = df.sort_values(["date", "tic"], ignore_index=True)

# Factorize the date index
df.index = df.date.factorize()[0]
df

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3
0,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3
0,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3
1,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4
1,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4
...,...,...,...,...,...,...,...,...
5280,2024-12-30,20.230000,20.309999,20.150000,20.309999,700,ZWS.TO,0
5280,2024-12-30,53.099998,53.360001,52.830002,53.299999,11600,ZWT.TO,0
5280,2024-12-30,10.510000,10.520000,10.430000,10.500000,168700,ZWU.TO,0
5280,2024-12-30,42.599998,42.610001,42.599998,42.610001,300,ZXM.TO,0


In [32]:

# Drop rows where any of the required columns have NaN values
df = df.dropna(subset=["close", "high", "low", "open", "tic", "day"])

# *Step 3: Add User Defined Features*
df["daily_return"] = df.close.pct_change(1)

In [33]:
df.head()

,date,close,high,low,open,volume,tic,day,daily_return
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3,NaN
0,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3,12.123810
0,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3,-0.782293
1,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4,0.632084
1,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4,44.075824


In [34]:
INDICATORS = ['macd',
               'rsi_30',
               'cci_30',
               'dx_30']

In [35]:
TICKERS = ["RY.TO", "TD.TO", "BNS.TO", "ENB.TO", "SHOP.TO"]  # Example TSX tickers

In [36]:
df = df[df['tic'].isin(TICKERS)]
df.shape

(23500, 9)

In [37]:
feature_engineer = FeatureEngineer(use_technical_indicator=True, tech_indicator_list=INDICATORS)


In [38]:
df = feature_engineer.add_technical_indicator(df)

In [39]:
df = feature_engineer.add_turbulence(df)

In [40]:
df["date"] = pd.to_datetime(df["date"])  # Convert main dataset date column

In [41]:
stock_dimension = len(df.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 5, State Space: 31


In [42]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5

}


In [43]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window,
                 validation_window=validation_window,
                 **env_kwargs)

In [44]:


DDPG_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 10_000,
                      "learning_rate": 0.0005,
                      "batch_size": 64
                    }

TD3_model_kwargs = {"batch_size": 100, "buffer_size": 1000000, "learning_rate": 0.0001}




timesteps_dict = {'a2c' : 0,
                 'ppo' : 0,
                 'ddpg' : 10_000,
                 'sac' : 0,
                 'td3' : 0
                 }

In [45]:
df_summary = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  # Not using A2C
    PPO_model_kwargs={},  # Not using PPO
    DDPG_model_kwargs=DDPG_model_kwargs,  # Using DDPG
    SAC_model_kwargs={},  # Not using SAC
    TD3_model_kwargs={},  # Not using TD3
    timesteps_dict=timesteps_dict
)

============Start Ensemble Strategy============
turbulence_threshold:  60.18625422507017
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_7
======a2c Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
a2c Sharpe Ratio:  -0.2228293066189291
======ddpg Training========
{'buffer_size': 10000, 'learning_rate': 0.0005, 'batch_size': 64}
Using cpu device
Logging to tensorboard_log/ddpg\ddpg_126_6
day: 1192, episode: 5
begin_total_asset: 1000000.00
end_total_asset: 4880368.29
total_reward: 3880368.29
total_cost: 998.99
total_trades: 2384
Sharpe: 1.141
-----------------------------------
| time/              |            |
|    episodes        | 4          |
|    fps             | 110        |
|    time_elapsed    | 43         |
|    total_timesteps | 4772       |
| train/             |            |
|    actor_loss      | 3.7e+03    |
|    critic_loss     | 2.06e+03   |
|    learning_

In [46]:
df_summary

,Iter,Val Start,Val End,Model Used,A2C Sharpe,PPO Sharpe,DDPG Sharpe,SAC Sharpe,TD3 Sharpe
0,126,2021-10-04 00:00:00,2022-01-05 00:00:00,SAC,-0.222829,-0.282167,-0.218307,0.4332,0.280615
1,189,2022-01-05 00:00:00,2022-04-05 00:00:00,SAC,-0.264447,-0.162491,-0.159215,-0.067648,-0.083847
2,252,2022-04-05 00:00:00,2022-07-06 00:00:00,TD3,-0.414646,-0.283824,-0.319578,-0.424588,-0.247861
3,315,2022-07-06 00:00:00,2022-10-05 00:00:00,PPO,0.104716,0.163186,-0.002351,0.007749,-0.000725


In [47]:
DDPG_model_kwargs_tuning = {
    "learning_rate": 0.0005,
    "buffer_size": 50000,
    "batch_size": 128,
    "learning_starts": 500,
    "tau": 0.005,
    "gamma": 0.99,
    "train_freq": (1, "step"),
    "gradient_steps": 1
}

In [48]:
df_summary_tuning = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  # Not using A2C
    PPO_model_kwargs={},  # Not using PPO
    DDPG_model_kwargs=DDPG_model_kwargs_tuning,  # Using DDPG
    SAC_model_kwargs={},  # Not using SAC
    TD3_model_kwargs={},  # Not using TD3
    timesteps_dict=timesteps_dict
)

============Start Ensemble Strategy============
turbulence_threshold:  60.18625422507017
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_8
======a2c Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
a2c Sharpe Ratio:  -0.07884967224883518
======ddpg Training========
{'learning_rate': 0.0005, 'buffer_size': 50000, 'batch_size': 128, 'learning_starts': 500, 'tau': 0.005, 'gamma': 0.99, 'train_freq': (1, 'step'), 'gradient_steps': 1}
Using cpu device
Logging to tensorboard_log/ddpg\ddpg_126_7
day: 1192, episode: 5
begin_total_asset: 1000000.00
end_total_asset: 1301450.83
total_reward: 301450.83
total_cost: 999.00
total_trades: 1192
Sharpe: 0.344
-----------------------------------
| time/              |            |
|    episodes        | 4          |
|    fps             | 126        |
|    time_elapsed    | 37         |
|    total_timesteps | 4772       |
| train/            

In [50]:
timesteps_dict = {'a2c' : 0,
                 'ppo' : 0,
                 'ddpg' : 200000,
                 'sac' : 0,
                 'td3' : 0
                 }

In [ ]:
df_summary_tuning = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  # Not using A2C
    PPO_model_kwargs={},  # Not using PPO
    DDPG_model_kwargs=DDPG_model_kwargs_tuning,  # Using DDPG
    SAC_model_kwargs={},  # Not using SAC
    TD3_model_kwargs={},  # Not using TD3
    timesteps_dict=timesteps_dict
)


============Start Ensemble Strategy============
turbulence_threshold:  60.18625422507017
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_9
======a2c Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
a2c Sharpe Ratio:  0.13500446633179955
======ddpg Training========
{'learning_rate': 0.0005, 'buffer_size': 50000, 'batch_size': 128, 'learning_starts': 500, 'tau': 0.005, 'gamma': 0.99, 'train_freq': (1, 'step'), 'gradient_steps': 1}
Using cpu device
Logging to tensorboard_log/ddpg\ddpg_126_8
day: 1192, episode: 5
begin_total_asset: 1000000.00
end_total_asset: 1470913.35
total_reward: 470913.35
total_cost: 998.98
total_trades: 3576
Sharpe: 0.511
-----------------------------------
| time/              |            |
|    episodes        | 4          |
|    fps             | 121        |
|    time_elapsed    | 39         |
|    total_timesteps | 4772       |
| train/             

In [ ]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 5e-4,  # Increase reward sensitivity
    "print_verbosity": 5
}